# Lesson 04 · Softmax & loss functions: turn 10 scores into probabilities, and measure how wrong the network is

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/04-ham-mat-mat-en.ipynb)

This notebook accompanies [Lesson 04 of Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/ham-mat-mat/), a deep-learning course originally in
Vietnamese. It is the English edition of the course's notebook
([Vietnamese original](https://github.com/trungnguyenvn/hoc-sau-notebooks/blob/main/04-ham-mat-mat.ipynb)):
the code, the data and every check are the same. The interactive demo is on the lesson page.
The course's MNIST network, 784 → 128 → 64 → 10, gives each image 10 scores (**logits**), one score per digit.
Using numpy, we write softmax ourselves to turn them into probabilities, and cross-entropy and MSE to measure how
wrong the network is; then we retrain the very network the demo uses, on 60,000 real MNIST images.

**What you'll do:**
1. Turn 10 logits into 10 probabilities with softmax, and write it so that it never overflows.
2. Compute cross-entropy directly from the logits, and understand why an untrained network has a loss close to
   $\ln 10$.
3. Retrain the course's network with cross-entropy, and get exactly the demo's 97.27%.
4. Compare MSE with cross-entropy on the “confidently wrong” images: first by their slopes, then by real training.
5. See how far softmax's probabilities can be trusted, and look at the loss as a landscape for Lesson 05.

**Background:** [Lesson 01 · Perceptron](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/perceptron/) ($z = xW + b$),
[Lesson 02 · Activation functions](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/ham-kich-hoat/) (ReLU) and
[Lesson 03 · Multilayer networks & the XOR problem](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/mlp-va-xor/)
(hidden layers). **Run time:** under a minute on a CPU, including the first MNIST download (~11 MB).

The notation follows the *Neural Computation* chapter of [MLSysBook](https://mlsysbook.ai/vol1/nn_computation/nn_computation.html),
with row vectors as in Lesson 01: $z = h W + b$. Every number in the notebook is computed by the cells below,
and matches the numbers in the demo.

> The **🤔 Predict first** cells ask a question before you run something. Answer it yourself, then run the next cell.

## 1 · Setup

Only numpy and matplotlib are needed (Colab has them preinstalled). The function `num()` prints numbers in the
English format: a decimal point, and commas between thousands. That is the same point Python uses when you *type*
a decimal (`0.1`); printing only adds the thousands separators and a real minus sign.

In [ ]:
import gzip
import hashlib
import itertools
import time
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import FuncFormatter

# The course's palette (colorblind-safe), always paired with a second encoding (solid/dashed lines, hatching, labels)
BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def num(x, digits=2):
    """A number in the English format: 13,007 · 0.1 · −4.38."""
    return f"{x:,.{digits}f}".replace("-", "−")


def pct(p, digits=2):
    return num(100 * p, digits) + "%"


def sci(x):
    """A very small number, in scientific notation: 1.2e−10."""
    return f"{x:.1e}".replace("-", "−")


def fmt_ticks(axis, digits=1):
    """matplotlib's axis tick labels in the same format as num()."""
    axis.set_major_formatter(FuncFormatter(lambda v, _: num(v, digits)))


print(num(60000, 0), num(np.log(10), 4), pct(0.9727), sci(1.23e-10))

## 2 · The MNIST data

MNIST (LeCun et al., 1998) has 70,000 grayscale 28×28 images of handwritten digits: 60,000 training images and
10,000 test images. The cell below downloads the 4 original files from a public mirror and checks their MD5 hashes,
so everyone who runs it gets exactly the same data. As in Lesson 01, each image is laid out as a row vector $x$ of
784 numbers in $[0, 1]$.

In [ ]:
URL = "https://ossci-datasets.s3.amazonaws.com/mnist/"
FILES = {  # MNIST's standard MD5 hashes
    "train-images-idx3-ubyte.gz": "f68b3c2dcbeaaa9fbdd348bbdeb94873",
    "train-labels-idx1-ubyte.gz": "d53e105ee54ea40749a09fcbcd1e9432",
    "t10k-images-idx3-ubyte.gz": "9fb629c4189551a2d022fa330f9573f3",
    "t10k-labels-idx1-ubyte.gz": "ec29112dd5afa0611ce80d1b7f02629c",
}
DATA = Path("mnist")


def fetch():
    DATA.mkdir(exist_ok=True)
    for name, md5 in FILES.items():
        p = DATA / name
        if not p.exists():
            urllib.request.urlretrieve(URL + name, p)
        assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} is corrupted; delete it and download it again"


def load(kind):
    """IDX format: the first 16 bytes of an image file (8 bytes of a label file) are the header; the rest are bytes
    0–255."""
    with gzip.open(DATA / f"{kind}-images-idx3-ubyte.gz") as f:
        images = np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 28, 28)
    with gzip.open(DATA / f"{kind}-labels-idx1-ubyte.gz") as f:
        labels = np.frombuffer(f.read(), np.uint8, offset=8)
    return images, labels


fetch()
train_img, train_lab = load("train")
test_img, test_lab = load("t10k")
Xtr, ytr = train_img.reshape(-1, 784) / 255.0, train_lab.astype(int)  # each row is one image
Xte, yte = test_img.reshape(-1, 784) / 255.0, test_lab.astype(int)
print(
    f"{num(len(ytr), 0)} training images + {num(len(yte), 0)} test images = {num(len(ytr) + len(yte), 0)};  X:",
    Xtr.shape,
)
print("labels:", np.unique(ytr))

The demo on the lesson page uses 18 real test images, split into three groups by how the trained network answers
them (chosen by the course's `reel/mnist_mlp/train.py`). The notebook reuses exactly those 18 images; the group names
will make sense in section 7, after we train the network ourselves. The figure below shows them, each titled with its
index in the test set and its true label.

In [ ]:
DEMO = {  # original indices in the MNIST test set, exactly as in the demo
    "correct & sure": [0, 1, 2, 3, 4, 7],
    "correct but unsure": [62, 340, 358, 447, 691, 890],
    "wrong & confident": [2654, 9009, 2939, 4880, 5955, 1112],
}
fig, axes = plt.subplots(3, 6, figsize=(8.5, 4.6))
for row, (group, idx) in zip(axes, DEMO.items()):
    for ax, i in zip(row, idx):
        ax.imshow(test_img[i], cmap="gray_r")
        ax.set_title(f"#{i} · {yte[i]}", fontsize=9)
        ax.axis("off")
    row[0].text(-0.3, 0.5, group, transform=row[0].transAxes, ha="right", va="center", fontsize=10)
plt.subplots_adjust(left=0.24, hspace=0.35)
plt.show()

## 3 · Ten scores: the logits of the 784 → 128 → 64 → 10 network

The course's network (MLSysBook uses exactly this network for MNIST) has two ReLU hidden layers and an output layer
of 10 neurons:

$$h_1 = \mathrm{ReLU}(x W_1 + b_1), \qquad h_2 = \mathrm{ReLU}(h_1 W_2 + b_2), \qquad z = h_2 W_3 + b_3 .$$

The ten numbers $z_0, \dots, z_9$ at the output layer are called **logits**: raw scores for each digit, not yet passed
through any function. The weights are initialized randomly with He's method (Lesson 09 will explain it):
$W \sim \mathcal{N}(0, 2/n_\text{in})$, $b = 0$, with seed 0, exactly as in the program that produced the demo.
The cell below prints the 10 logits of the *untrained* network for test image #0.

In [ ]:
SIZES = [784, 128, 64, 10]


def init(seed=0, last_scale=1.0):
    """He initialization, as in reel/mnist_mlp/train.py. Also returns rng, because training uses it to shuffle the
    images."""
    rng = np.random.default_rng(seed)
    params = [
        [rng.normal(0, np.sqrt(2 / n_in), (n_in, n_out)), np.zeros(n_out)] for n_in, n_out in itertools.pairwise(SIZES)
    ]
    params[2][0] *= last_scale  # scale the output layer's W₃ up or down: only for the experiments in sections 5 and 8
    return params, rng


def forward(params, X):
    """Each row of X is one image; returns h₁, h₂ and z (10 logits per image)."""
    (W1, b1), (W2, b2), (W3, b3) = params
    h1 = np.maximum(0, X @ W1 + b1)  # ReLU (Lesson 02)
    h2 = np.maximum(0, h1 @ W2 + b2)
    return h1, h2, h2 @ W3 + b3


params0, _ = init(seed=0)
n_params = sum(W.size + b.size for W, b in params0)
z0 = forward(params0, Xte[:1])[2][0]  # test image #0
print(f"the network has {num(n_params, 0)} parameters; image #0 is a {yte[0]}")
print("10 logits:", "  ".join(f"{k}: {num(v, 3)}" for k, v in enumerate(z0)))
print(f"largest logit at digit {z0.argmax()}; sum of the 10 logits = {num(z0.sum(), 2)}")
assert n_params == 109386  # 109,184 weights (exactly the number of MACs in Lesson 01) + 202 biases

Logits can be negative or positive, and their sum means nothing. The network is untrained, so which digit gets the
highest score is just chance. To read these 10 scores as “the probability with which the network thinks the image
is digit $k$”, we need 10 **positive** numbers that **sum to 1**.

### 🤔 Predict first
The simplest way: divide each logit by the sum, $z_k / \sum_j z_j$. The 10 new numbers are sure to sum to 1.
Are they 10 probabilities? What “probability” will the digit with the largest logit (the 9) get?

In [ ]:
naive = z0 / z0.sum()
print("z / Σz =", "  ".join(f"{k}: {num(v, 3)}" for k, v in enumerate(naive)))
print(
    f"sum = {num(naive.sum(), 3)};  smallest {num(naive.min(), 3)} (digit {naive.argmin()});  largest {num(naive.max(), 3)}"
)

The sum is indeed 1, but some numbers are negative and some are larger than 1: these are not probabilities. Worse,
the sum of the 10 logits is negative here, so the division **reverses the order**: the 9, which has the largest
logit, gets the smallest “probability” (−1.516). We need a way to turn every logit into a positive number before
dividing, while keeping the order: that is the exponential function.

## 4 · Softmax: exponentiate, then divide by the sum

$$\hat y_k = \mathrm{softmax}(z)_k = \frac{e^{z_k}}{\sum_{j=0}^{9} e^{z_j}}$$

$e^{z}$ is always positive and increases with $z$, so a larger logit always gets a larger probability, and no class
is forced to exactly 0. Written as a loop, the three steps are explicit:

In [ ]:
def softmax_loop(z):
    e = [float(np.exp(v)) for v in z]  # 1. exponentiate: every number becomes positive, the order is kept
    total = sum(e)  # 2. add them up
    return [v / total for v in e]  # 3. divide by the sum: the new sum is 1


p0 = softmax_loop(z0)
print("softmax(z) =", "  ".join(f"{k}: {num(v, 3)}" for k, v in enumerate(p0)))
print(f"sum = {num(sum(p0), 3)};  largest {num(max(p0), 3)} (digit {int(np.argmax(p0))});  smallest {num(min(p0), 3)}")

The network is untrained, so the 10 probabilities only range from 0.057 to 0.205, around 0.1: it is almost guessing
uniformly. The 9 still comes first, following the order of the logits.

Softmax depends only on the **differences** between the logits: add the same number $c$ to all 10 logits and $e^{c}$
appears in both the numerator and the denominator, and cancels. Multiplying the logits by a number is different: the
differences between them stretch, and the largest probability goes toward 1. (That is why Goodfellow et al. argue
that a more accurate name would be “softargmax”: a *soft* version of picking the digit with the highest score.)
The numpy version below handles many images at once, one image per row:

In [ ]:
def softmax(Z):
    """Row-wise softmax, the stable version: subtract the largest logit before exponentiating (the next cell explains
    why)."""
    E = np.exp(Z - Z.max(axis=-1, keepdims=True))
    return E / E.sum(axis=-1, keepdims=True)


assert np.allclose(softmax(z0), p0)
for name, z in [("z", z0), ("z + 100", z0 + 100), ("z − 5", z0 - 5), ("3·z", 3 * z0), ("10·z", 10 * z0)]:
    p = softmax(z)
    print(f"{name:>8}: largest probability {num(p.max(), 4)} (digit {p.argmax()}), smallest {num(p.min(), 4)}")
assert np.allclose(softmax(z0 + 100), softmax(z0)) and np.allclose(softmax(z0 - 5), softmax(z0))

### Why subtract the largest logit?
A computer can only represent real numbers up to some limit. $e^{1000}$ exceeds that limit and becomes `inf`
(infinity), and `inf / inf` is `nan` (“not a number”). Since softmax doesn't change when the same number is added to
every logit, we subtract the largest logit first: the largest exponential is then $e^0 = 1$, which never overflows.
The cell below compares the two ways on three very large logits, then prints the overflow threshold of three common
floating-point types.

In [ ]:
z_big = np.array([1000.0, 999.0, 990.0])
with np.errstate(over="ignore", invalid="ignore"):  # silence numpy's warnings, to look straight at the result
    e = np.exp(z_big)
    print("naive way:           e^z =", e, "  →  e^z / Σe^z =", e / e.sum())
print("subtract max first: ", "  ".join(num(v, 6) for v in softmax(z_big)))
for t in (np.float16, np.float32, np.float64):
    print(f"{t.__name__:>8}: e^z overflows when z > {num(float(np.log(np.finfo(t).max)), 2)}")

With float64 (numpy's default), it takes $z > 709.78$ to overflow, but when training on GPUs people often compute in
float16, where a single logit larger than 11.09 is enough. In section 7 we will see that the real logits of the
trained network go far beyond that threshold.

## 5 · Cross-entropy: the “surprise” at seeing the correct answer

To learn, the network needs a number that says how wrong it is: the **loss function**, the smaller the better. With
one-hot labels ($y_k = 1$ at the correct digit $c$, 0 at the other digits), cross-entropy has only one term left:

$$\mathcal{L}_\text{CE} = -\sum_{k} y_k \ln \hat y_k = -\ln \hat y_c .$$

$-\ln \hat y_c$ is the network's **surprise** at seeing the answer: assign 0.9 to the correct digit and there is
hardly any surprise; assign 0.0001 and the surprise is large. The natural logarithm (ln) measures it in units of
*nats*. The $\ln$ of softmax can be written compactly as $\ln \hat y_k = z_k - \ln \sum_j e^{z_j}$ (**log-softmax**),
so we compute it directly from the logits, again subtracting the max first. That way we never have to take $\ln 0$.

In [ ]:
def log_softmax(Z):
    """ln softmax(Z) computed directly from the logits: (z − max) − ln Σ e^(z − max)."""
    Zs = Z - Z.max(axis=-1, keepdims=True)
    return Zs - np.log(np.exp(Zs).sum(axis=-1, keepdims=True))


def cross_entropy(Z, y):
    """The mean over the images of −ln ŷ_c, where c is each image's correct digit."""
    return float(-log_softmax(Z)[np.arange(len(y)), y].mean())


c0 = yte[0]
print(f"image #0 is a {c0}: ŷ_{c0} = {num(p0[c0], 4)}  →  −ln ŷ_{c0} = {num(cross_entropy(z0[None], yte[:1]), 4)}")
for p in [0.99, 0.9, 0.8, 0.5, 0.1, 0.01, 0.0001]:
    print(f"probability of the correct answer {num(p, 4):>6}  →  loss {num(-np.log(p), 3)}")

The closer the probability of the correct answer gets to 0, the larger the loss, with no upper bound:
$-\ln \hat y_c \to \infty$ as $\hat y_c \to 0$. (MLSysBook uses exactly the examples 0.8 → 0.223 and 0.9 → 0.105.)
On a dataset, the loss is the average over the images.

### 🤔 Predict first
The *untrained* network of section 3. Over all 10,000 test images, what number is the mean cross-entropy close to?
Hint: what probability does a network that guesses all 10 digits uniformly give the correct answer?

In [ ]:
loss0 = cross_entropy(forward(params0, Xte)[2], yte)
print(f"loss before training: {num(loss0, 4)}   ·   −ln(1/10) = ln 10 = {num(np.log(10), 4)}")
assert abs(loss0 - 2.4257) < 5e-4, "differs from the demo: check the data"

A uniform guess gives $\hat y_c = 1/10$ and a loss of exactly $\ln 10 \approx 2.3026$: that is the “knows nothing”
baseline of every 10-class classification problem. The untrained network gets 2.4257 (exactly the number the demo
shows), slightly higher, because random logits are not perfectly equal, so the softmax is not quite uniform.
Karpathy recommends always checking this number before training: if the initial loss is far from $\ln 10$, the
output layer is being initialized too “confidently”. Let's try scaling the output layer's $W_3$ up and down:

In [ ]:
for s in [0, 0.1, 0.5, 1, 2, 4]:
    Zs = forward(init(seed=0, last_scale=s)[0], Xte)[2]
    print(
        f"W₃ × {num(s, 1):>3}: logit std {num(Zs.std(), 2)}, loss {num(cross_entropy(Zs, yte), 4)}, "
        f"mean −ln ŷₖ over all 10 classes {num(-log_softmax(Zs).mean(), 4)}"
    )
    if s == 0:
        assert np.isclose(cross_entropy(Zs, yte), np.log(10))  # W₃ = 0: every logit is 0, a perfectly uniform guess

With $W_3 = 0$, every logit is 0 and the loss is exactly $\ln 10$. The more spread out the logits, the more randomly
“confident” the network is, and the higher the initial loss: 3.3089 when $W_3$ is 4 times larger. The last column is
the mean of $-\ln \hat y_k$ over all 10 classes, not just the correct one. It is never below $\ln 10$, because by the
inequality of arithmetic and geometric means, $\ln \sum_j e^{z_j} \ge \ln 10 + \frac{1}{10}\sum_j z_j$, with equality
only when all the logits are equal.

### The slope of cross-entropy: $\hat y - y$
To know which way to adjust the logits, we need the derivative of the loss with respect to each logit. With softmax
+ cross-entropy, the result is surprisingly compact (proved in Lesson 06; MLSysBook states it too):

$$\frac{\partial \mathcal{L}_\text{CE}}{\partial z_k} = \hat y_k - y_k .$$

We check it with **finite differences**: nudge each logit by a small $\pm\varepsilon$ and see how much the loss
changes.

In [ ]:
def grad_ce(Z, y):
    """∂𝓛/∂z of the cross-entropy averaged over the images: (ŷ − y_onehot) / number of images."""
    G = softmax(Z)
    G[np.arange(len(y)), y] -= 1
    return G / len(y)


def numeric_grad(f, z, eps=1e-6):
    """The derivative of f with respect to each logit, by central differences: (f(z + ε) − f(z − ε)) / 2ε."""
    g = np.zeros_like(z)
    for k in range(len(z)):
        d = np.zeros_like(z)
        d[k] = eps
        g[k] = (f(z + d) - f(z - d)) / (2 * eps)
    return g


g0 = grad_ce(z0[None], yte[:1])[0]
g0_num = numeric_grad(lambda z: cross_entropy(z[None], yte[:1]), z0)
print("ŷ − y =", "  ".join(f"{k}: {num(v, 3)}" for k, v in enumerate(g0)))
print("largest deviation from the finite differences:", sci(np.abs(g0 - g0_num).max()))
assert np.allclose(g0, g0_num, atol=1e-8)

At the correct digit (the 7), the slope is $\hat y_7 - 1 < 0$: increasing this logit decreases the loss. At the
other 9 digits, the slope is $\hat y_k > 0$: decreasing them decreases the loss, and whichever digit currently has a
high probability gets pushed down the hardest. Cross-entropy only *looks at* $\hat y_c$, but $\hat y_c$ depends on
all 10 logits, so all 10 get a push. Each logit's slope always lies in $(-1, 1)$, even when the loss is very large.

## 6 · Training with cross-entropy: rebuilding the demo's network

The course's recipe: SGD on mini-batches of 64 images, learning rate $\eta = 0.1$, 5 epochs over the 60,000 images,
seed 0, exactly as in the program that produced the demo. Each step computes the logits for 64 images, takes
$\partial\mathcal{L}/\partial z$, propagates it backward through the layers (Lesson 06), then subtracts $\eta$ times
the gradient from every weight (Lesson 05). In the function below, only one line depends on the loss function:
`dz = grad(Z, y)`. We will change exactly that line in section 8.

In [ ]:
def train(seed=0, eta=0.1, epochs=5, grad=grad_ce, last_scale=1.0, X=Xtr, y=ytr, X_eval=Xte, y_eval=yte):
    """SGD on mini-batches of 64 images, as in reel/mnist_mlp/train.py. grad(Z, y) returns ∂𝓛/∂z of the loss function
    in use."""
    params, rng = init(seed, last_scale)
    log = {"steps": 0, "step_loss": [], "epoch_acc": [], "epoch_loss": []}
    running = []
    for _ in range(epochs):
        order = rng.permutation(len(y))
        for i in range(0, len(order), 64):
            idx = order[i : i + 64]
            h1, h2, Z = forward(params, X[idx])
            running.append(cross_entropy(Z, y[idx]))  # always record CE, even if the network learns with another loss
            if len(running) == 50:  # record the mean once every 50 steps
                log["step_loss"].append(float(np.mean(running)))
                running = []
            dz = grad(Z, y[idx])  # the only line that depends on the loss function
            W2, W3 = params[1][0], params[2][0]  # backpropagation (Lesson 06) ...
            dh2 = (dz @ W3.T) * (h2 > 0)
            dh1 = (dh2 @ W2.T) * (h1 > 0)
            grads = [(X[idx].T @ dh1, dh1.sum(0)), (h1.T @ dh2, dh2.sum(0)), (h2.T @ dz, dz.sum(0))]
            for layer, (gW, gb) in zip(params, grads):  # ... and one gradient descent step (Lesson 05)
                layer[0] -= eta * gW
                layer[1] -= eta * gb
            log["steps"] += 1
        Z_eval = forward(params, X_eval)[2]
        log["epoch_acc"].append(float((Z_eval.argmax(1) == y_eval).mean()))
        log["epoch_loss"].append(cross_entropy(Z_eval, y_eval))
    return params, log


t0 = time.perf_counter()
params, log = train(seed=0)
print(f"{num(log['steps'], 0)} steps ({num(time.perf_counter() - t0, 1)} s)")
for e, (a, l) in enumerate(zip(log["epoch_acc"], log["epoch_loss"]), 1):
    print(f"epoch {e}: test accuracy {pct(a)}, loss {num(l, 4)}")
acc, loss = log["epoch_acc"][-1], log["epoch_loss"][-1]
print(f"training loss: first 50 steps {num(log['step_loss'][0], 4)}, last 50 steps {num(log['step_loss'][-1], 4)}")
assert abs(acc - 0.9727) < 5e-4 and abs(loss - 0.0865) < 5e-4, "differs from the demo: check the data"
assert np.allclose(log["epoch_acc"], [0.9427, 0.9633, 0.9688, 0.9706, 0.9727], atol=1e-3)
assert abs(log["step_loss"][0] - 1.2031) < 1e-3 and abs(log["step_loss"][-1] - 0.059) < 1e-3

97.27% correct on the 10,000 test images, mean loss 0.0865: the numbers the demo shows. The figure below plots the
loss during training: the solid blue line is the cross-entropy on the training images (averaged every 50 steps), the
orange squares joined by a dashed line are the loss on the test set after each epoch, and the gray line is the
$\ln 10$ baseline of uniform guessing.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4))
steps = 50 * np.arange(1, len(log["step_loss"]) + 1)
ax.plot(steps, log["step_loss"], color=BLUE, lw=2, label="training (every 50 steps)")
per_epoch = log["steps"] // 5 * np.arange(1, 6)
ax.plot(per_epoch, log["epoch_loss"], color=ORANGE, ls="--", marker="s", ms=7, label="test (after each epoch)")
ax.axhline(np.log(10), color=GREY, lw=1, ls=":")
ax.text(
    steps[-1], np.log(10) - 0.06, "ln 10: uniform guess over 10 digits", ha="right", va="top", color="#555", fontsize=9
)
ax.plot([0], [loss0], marker="o", color=GREY)
ax.annotate(f"untrained: {num(loss0, 2)}", (0, loss0), xytext=(120, loss0 - 0.03), fontsize=9, color="#555")
ax.set(xlabel="SGD step (64 images per step)", ylabel="cross-entropy", ylim=(0, 2.6), xlim=(-50, steps[-1] + 50))
ax.set_title("Loss while training the 784 → 128 → 64 → 10 network (η = 0.1, seed 0)")
fmt_ticks(ax.yaxis)
fmt_ticks(ax.xaxis, 0)
ax.legend(frameon=False)
ax.grid(axis="y", alpha=0.3)
plt.show()

The loss drops very fast in the first few hundred steps (the mean of the very first 50 steps is already 1.2031),
then decreases more and more slowly. In the end the training loss is lower than the test loss: the network has
started to memorize the training set a little (Lesson 08).

## 7 · Softmax on the trained network's real logits

Now we look at the demo's 18 images again, through the network we just trained. The cell below computes the logits
and probabilities for the whole test set, checks that each image falls into the group the demo put it in (“correct
& sure”: correct and $0.9 \le \hat y_c < 0.999$; “unsure”: correct but $\hat y_c < 0.6$; “wrong & confident”: wrong,
with $\ge 90\%$ for another digit), then prints a few images.

In [ ]:
Z = forward(params, Xte)[2]
P = softmax(Z)
pred, p_true, p_max = Z.argmax(1), P[np.arange(len(yte)), yte], P.max(1)
# Two of the demo's images lie close to a threshold (#3: ŷ_c = 0.99897; one wrong image has p_max = 0.9006). On another
# machine, additions round differently, so the logits can differ by ~0.01; TOL allows that small difference without
# raising an error.
TOL = 0.002
rules = {
    "correct & sure": (pred == yte) & (p_true >= 0.9 - TOL) & (p_true < 0.999 + TOL),
    "correct but unsure": (pred == yte) & (p_true < 0.6 + TOL),
    "wrong & confident": (pred != yte) & (p_max >= 0.9 - TOL),
}
for group, idx in DEMO.items():
    assert rules[group][idx].all(), f"group “{group}” differs from the demo"
    print(f"{group}:")
    for i in idx[:3]:
        print(
            f"  #{i:>4} digit {yte[i]}: the network picks {pred[i]} with {pct(p_max[i], 2)}; "
            f"ŷ of the correct answer {pct(p_true[i], 3)}, CE = {num(-np.log(p_true[i]), 4)}"
        )
z_2654 = [-6.47, 10.9505, 1.7756, -0.2557, 0.6202, -2.619, 1.5918, -1.8769, 2.7477, -3.0513]  # logits in the demo
assert np.abs(Z[2654] - z_2654).max() < 0.01, "differs from the demo: check the data"

The figure below puts three images side by side, one image per row: the logits (middle) and the probabilities after
softmax (right). The blue bar is the correct digit; the hatched orange bar is the digit the network picks when it
picks wrong.

In [ ]:
def bars(ax, values, c, wrong, fmt):
    colors = [BLUE if k == c else ORANGE if k == wrong else GREY for k in range(10)]
    hatch = ["//" if k == wrong else "" for k in range(10)]
    ax.barh(range(10), values, color=colors, hatch=hatch, edgecolor="white")
    ax.set_yticks(range(10))
    ax.invert_yaxis()
    for k, v in enumerate(values):
        if k in (c, wrong):
            ax.text(v, k, " " + fmt(v), va="center", ha="left" if v >= 0 else "right", fontsize=8)


fig, axes = plt.subplots(3, 3, figsize=(10, 7.6), gridspec_kw={"width_ratios": [1, 2, 2]})
for row, i in zip(axes, [0, 340, 2654]):
    c, k = yte[i], pred[i]
    row[0].imshow(test_img[i], cmap="gray_r")
    row[0].set_title(f"#{i} · digit {c}", fontsize=10)
    row[0].axis("off")
    bars(row[1], Z[i], c, k if k != c else -1, lambda v: num(v, 1))
    row[1].axvline(0, color="black", lw=0.8)
    row[1].set(xlim=(-14, 17), ylabel="digit")
    fmt_ticks(row[1].xaxis, 0)
    bars(row[2], P[i], c, k if k != c else -1, lambda v: pct(v, 3 if v < 0.01 else 1))
    row[2].set(xlim=(0, 1.3), xticks=[0, 0.5, 1])
    fmt_ticks(row[2].xaxis)
axes[0, 1].set_title("logits z")
axes[0, 2].set_title("probabilities ŷ = softmax(z)")
axes[2, 1].set_xlabel("logit")
axes[2, 2].set_xlabel("probability")
plt.tight_layout()
plt.show()

- **#0 (a 7):** the logit of the 7 is far above those of the other digits, so almost all of the probability goes to
  the 7. The loss is close to 0 but not 0: raising the logit of the 7 further still lowers the cross-entropy, even
  though the accuracy doesn't change.
- **#340 (a 5):** the logits of the 5 and of the 3 are almost equal, so the probability is split in two. The network
  picks correctly, and accuracy counts this image as fully correct, but cross-entropy still penalizes it because the
  network is unsure.
- **#2654 (a 6):** the network is almost absolutely sure this is a 1. The probability of the 6 is tiny, so the loss
  is very large.

The trained network's logits are much larger than before training. Remember the float16 overflow threshold of
section 4:

In [ ]:
print(
    f"largest logit on the test set: {num(Z.max(), 2)};  images whose largest logit > 11.09: {num((Z.max(1) > 11.09).sum(), 0)}"
)
Z16 = Z.astype(np.float16)
with np.errstate(over="ignore", invalid="ignore"):
    E16 = np.exp(Z16)
    P_naive = E16 / E16.sum(1, keepdims=True)
print(f"naive softmax in float16: {num(np.isnan(P_naive).any(1).sum(), 0)} images give nan")
ce16 = float(-log_softmax(Z16)[np.arange(len(yte)), yte].astype(float).mean())
print(f"log-softmax in float16: loss {num(ce16, 4)}  (float64: {num(loss, 4)})")
assert np.isnan(P_naive).any() and abs(ce16 - loss) < 1e-3

More than half of the test set would make the naive softmax give `nan` if computed in float16. The version that
subtracts the max, in the same number type, gives a loss almost equal to float64's. That is why libraries always take
*logits*, not probabilities: PyTorch's `cross_entropy` computes the log-softmax internally, and when training with
automatic float16, PyTorch still computes softmax, log-softmax and the loss functions in float32
([autocast documentation](https://docs.pytorch.org/docs/stable/amp.html)).

## 8 · MSE or cross-entropy?

Squared error is the familiar loss function of regression problems. Applied to the 10 probabilities (as the demo
does): add up the squared deviations over the 10 classes, then average over the images,

$$\mathrm{MSE} = \sum_{k=0}^{9} (\hat y_k - y_k)^2 .$$

Each deviation $|\hat y_k - y_k|$ is at most 1 and the $\hat y_k$ sum to 1, so the MSE of one image is never more
than 2 (when all of the probability goes to one wrong digit). The slope with respect to the logits can also be
computed from the derivative of softmax; with respect to the logit of the correct digit, it simplifies to
$\partial \mathrm{MSE}/\partial z_c = -2\,\hat y_c\,\mathrm{MSE}$.

In [ ]:
def mse(Z, y):
    R = softmax(Z)
    R[np.arange(len(y)), y] -= 1  # ŷ − y_onehot
    return float((R**2).sum(1).mean())


def grad_mse(Z, y):
    """∂MSE/∂z_j = 2·ŷ_j·(r_j − Σₖ rₖŷₖ), with r = ŷ − y (the derivative of softmax), divided by the number of
    images."""
    P_ = softmax(Z)
    R = P_.copy()
    R[np.arange(len(y)), y] -= 1
    return 2 * P_ * (R - (R * P_).sum(1, keepdims=True)) / len(y)


i = 2654
g_num = numeric_grad(lambda z: mse(z[None], yte[i : i + 1]), Z[i])
assert np.allclose(grad_mse(Z[i : i + 1], yte[i : i + 1])[0], g_num, atol=1e-8)
mse_test = mse(Z, yte)
print(f"MSE on the test set: {num(mse_test, 4)}   (cross-entropy: {num(loss, 4)})")
assert abs(mse_test - 0.0411) < 5e-4, "differs from the demo: check the data"

Now compare the two functions on the demo's 6 “wrong & confident” images: the loss, and the slope with respect to
the logit of the correct digit, that is, the push each function gives that logit.

In [ ]:
print("image   true→pred     ŷ_c      CE     MSE   CE slope   MSE slope   CE/MSE")
for i in DEMO["wrong & confident"]:
    z, c = Z[i : i + 1], yte[i : i + 1]
    d_ce, d_mse = grad_ce(z, c)[0, c[0]], grad_mse(z, c)[0, c[0]]
    m = mse(z, c)
    assert np.isclose(d_mse, -2 * p_true[i] * m)  # the compact formula above
    print(
        f"#{i:>4}   {yte[i]} → {pred[i]}   {sci(p_true[i]):>9}  {num(cross_entropy(z, c), 2):>6}  {num(m, 4)}  "
        f"{num(d_ce, 4):>9}  {num(d_mse, 5):>10}  {num(d_ce / d_mse, 0):>6}×"
    )

All 6 images have an MSE close to the ceiling of 2, so MSE can hardly tell “badly wrong” from “very badly wrong”.
Cross-entropy ranges from 5.86 to 9.36 and has no ceiling. The slope matters more: with cross-entropy, the logit of
the correct digit gets a push close to $-1$, the largest possible. With MSE, the push is 88 to 2,903 times smaller,
because $-2\,\hat y_c\,\mathrm{MSE}$ has the factor $\hat y_c \approx 0$.

The figure below drags the logit of the correct digit (the 6) of image #2654 from very low to very high, keeping
the other 9 logits fixed, as the demo does. On the left is the loss: cross-entropy (blue, solid), MSE (orange,
dashed) and the 0–1 error (green, dotted: 1 for a wrong prediction, 0 for a correct one). On the right is the
magnitude of the slope with respect to that logit, on a logarithmic vertical axis.

In [ ]:
c = yte[2654]
grid = np.linspace(-10, 20, 601)
Zg = np.repeat(Z[2654][None], len(grid), axis=0)
Zg[:, c] = grid
yg = np.full(len(grid), c)
ce_g = -log_softmax(Zg)[:, c]
R = softmax(Zg) - np.eye(10)[c]
mse_g = (R**2).sum(1)
rival = np.delete(Z[2654], c).max()  # the largest logit of the other 9 digits
fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.6))
a.plot(grid, ce_g, color=BLUE, lw=2.2, label="cross-entropy")
a.plot(grid, mse_g, color=ORANGE, lw=2, ls="--", label="MSE")
a.plot(grid, (grid < rival).astype(float), color=GREEN, lw=2, ls=":", label="0–1 error")
b.semilogy(grid, np.abs(grad_ce(Zg, yg)[:, c] * len(grid)), color=BLUE, lw=2.2, label="cross-entropy")
b.semilogy(grid, np.abs(grad_mse(Zg, yg)[:, c] * len(grid)), color=ORANGE, lw=2, ls="--", label="MSE")
for ax in (a, b):
    ax.axvline(Z[2654, c], color="black", lw=1)
    ax.text(Z[2654, c] + 0.4, 0.97, "actual network", transform=ax.get_xaxis_transform(), va="top", fontsize=9)
    ax.set_xlabel(f"logit of the correct digit ({c}); the other 9 logits fixed")
    fmt_ticks(ax.xaxis, 0)
    ax.legend(frameon=False, fontsize=9)
a.set(ylabel="loss", ylim=(0, 12), title=f"Image #2654: loss vs. the logit of the {c}")
fmt_ticks(a.yaxis, 0)
b.set(ylabel="|slope| (log scale)", ylim=(1e-9, 3), title="Size of the push on that logit")
plt.tight_layout()
plt.show()

Left: while the logit of the 6 is still below the logit of the 1, cross-entropy rises almost along a straight line
(about 1 nat for each unit of logit), while MSE lies flat near 2. Right: the slope of cross-entropy stays close to 1
throughout the wrong-prediction region, while the slope of MSE falls exponentially as the logit decreases. The 0–1
error is completely flat everywhere except for one jump: it gives no gradient, so it can't be used for learning,
even though it is exactly what we want to reduce.

### A network that starts out confidently wrong
On the trained network, very few images fall deep into MSE's flat region (the cell below counts them). To see what
happens when very many images lie there, we initialize the output layer 30 times larger: the random logits are so
spread out that the untrained network is already “certain” of a random digit for most of the images. The cell below
counts the images with $\hat y_c < 0.001$ at the start.

In [ ]:
print(f"trained network of section 6: {(p_true < 1e-3).sum()} of the {num(len(yte), 0)} test images have ŷ_c < 0.001")
params30, _ = init(seed=0, last_scale=30)
Z30 = forward(params30, Xtr)[2]
stuck0 = (softmax(Z30)[np.arange(len(ytr)), ytr] < 1e-3).sum()
print(f"W₃ × 30: initial loss {num(cross_entropy(forward(params30, Xte)[2], yte), 2)} (versus ln 10 = 2.30)")
print(
    f"{num(stuck0, 0)} of the {num(len(ytr), 0)} training images have ŷ_c < 0.001 at the start ({pct(stuck0 / len(ytr), 1)})"
)

### 🤔 Predict first
Train this network for one epoch (same η = 0.1, same 60,000 images), once with cross-entropy, once with MSE.
Afterwards, how many of the 10,000 test images still have $\hat y_c < 0.001$? Tens, hundreds or thousands?

In [ ]:
stuck = {"ce": [], "mse": []}
for name, g in [("ce", grad_ce), ("mse", grad_mse)]:
    for seed in range(3):
        p_s, log_s = train(seed=seed, epochs=1, grad=g, last_scale=30)
        pc = softmax(forward(p_s, Xte)[2])[np.arange(len(yte)), yte]
        stuck[name].append(int((pc < 1e-3).sum()))
        print(
            f"{name:>3}, seed {seed}: test accuracy {pct(log_s['epoch_acc'][0], 2)}, "
            f"{num(stuck[name][-1], 0):>5} images still have ŷ_c < 0.001"
        )
print(f"MSE leaves at least {num(min(stuck['mse']) / max(stuck['ce']), 0)} times as many stuck images as cross-entropy")
assert max(stuck["ce"]) * 10 < min(stuck["mse"])

After one epoch, cross-entropy leaves only 57–73 test images still stuck at $\hat y_c < 0.001$. MSE leaves
3,512–4,122 images, at least 48 times as many: a gap far larger than the variation between seeds. MSE's accuracy is
also lower for all three seeds (57.22–63.49% versus 65.71–89.41%), even though cross-entropy varies very strongly
between seeds. The overconfident initialization hurts cross-entropy too (the first epoch in section 6 reached
94.27%), but it gets out.

Bridle (1990) pointed this out, and the *Deep Learning* book (Goodfellow et al., section 6.2.2.3) repeats it: with
softmax, squared error can fail to correct the network even when it is very confidently wrong, because there is no
logarithm to “undo” the exponential. Cross-entropy has one: $-\ln \hat y_c = \ln \sum_j e^{z_j} - z_c$ is almost
linear in the logits when the network is wrong, so the slope never vanishes.

## 9 · How far can softmax's probabilities be trusted?

Softmax outputs numbers that look like probabilities. If the network says 90%, is it right about 90% of the time?
The cell below splits the 10,000 test images by their largest probability (the network's “confidence”), then
compares the mean confidence with the actual accuracy in each group.

In [ ]:
correct = pred == yte
edges = [0, 0.5, 0.6, 0.7, 0.8, 0.9, 0.99, 1.0]
bins = []
for lo, hi in itertools.pairwise(edges):
    m = (p_max >= lo) & ((p_max < hi) | (hi == 1.0))
    bins.append((lo, hi, int(m.sum()), p_max[m].mean(), correct[m].mean()))
    print(
        f"confidence {pct(lo, 0):>4}–{pct(hi, 0):<4}: {num(m.sum(), 0):>5} images, mean confidence {pct(p_max[m].mean())}, correct {pct(correct[m].mean())}"
    )
sure, wrong_sure = p_max >= 0.99, ~correct & (p_max >= 0.9)
print(f"whole set: mean confidence {pct(p_max.mean())}, correct {pct(correct.mean())}")
print(
    f"{num(sure.sum(), 0)} images get ≥ 99%, of which {pct(correct[sure].mean())} are correct; "
    f"{wrong_sure.sum()} of the {(~correct).sum()} errors give ≥ 90% to a wrong digit"
)
# counts on a float-trained network: allow a difference of a few images between machines (BLAS rounds differently)
assert abs(sure.sum() - 7799) <= 5 and abs(correct[sure].mean() - 0.9986) < 1e-3, "differs from the demo"
assert abs(wrong_sure.sum() - 52) <= 5 and abs((~correct).sum() - 273) <= 5, "differs from the demo"

The figure below plots the table above: each blue dot is a group, the horizontal axis is the mean confidence, the
vertical axis the accuracy, and the number next to each dot is its number of images. The dashed gray diagonal is
“says 90%, right 90% of the time”.

In [ ]:
fig, ax = plt.subplots(figsize=(4.8, 4.6))
ax.plot([0, 1], [0, 1], color=GREY, ls="--", lw=1)
cx, cy = [b[3] for b in bins], [b[4] for b in bins]
ax.scatter(cx, cy, s=[30 + 120 * np.sqrt(b[2] / len(yte)) for b in bins], color=BLUE, zorder=3)
for b in bins:
    # the last two dots sit close together in the top right corner: the last dot's label goes below it, to the right
    last = b[3] > 0.99
    ax.annotate(
        f"{num(b[2], 0)} images",
        (b[3], b[4]),
        xytext=(16, -18) if last else (-8, 6),
        textcoords="offset points",
        ha="center" if last else "right",
        fontsize=8,
    )
ax.set(xlim=(0.3, 1.02), ylim=(0.3, 1.02), xlabel="mean confidence (largest probability)", ylabel="accuracy")
ax.set_title("Confidence and accuracy\n(10,000 test images)", fontsize=10)
fmt_ticks(ax.xaxis)
fmt_ticks(ax.yaxis)
ax.set_aspect("equal")
plt.show()

On real digit images, the dots lie close to the diagonal: mean confidence 97.15%, accuracy 97.27%. This small
network is “honest” on the kind of data it has learned. (Guo et al., 2017, found that modern, much larger networks
are often overconfident: their dots lie below the diagonal.) But 52 errors still get ≥ 90% from the network: high
confidence does not guarantee a correct answer.

### 🤔 Predict first
Give the network 1,000 **noise** images: each pixel is a uniform random number in $[0, 1]$, not a digit at all.
The network has never seen images like these. Will its largest probability be around 1/10 (“don't know”), or still
high?

In [ ]:
noise = np.random.default_rng(0).random((1000, 784))
Pn = softmax(forward(params, noise)[2])
print(
    f"{num(len(noise), 0)} noise images: mean largest probability {pct(Pn.max(1).mean())};  {pct((Pn.max(1) >= 0.9).mean(), 1)} of the noise images get ≥ 90%"
)
print("digits picked:", {k: int(n) for k, n in enumerate(np.bincount(Pn.argmax(1), minlength=10)) if n})
fig, axes = plt.subplots(1, 5, figsize=(8, 1.9))
for ax, j in zip(axes, np.argsort(-Pn.max(1))[:5]):
    ax.imshow(noise[j].reshape(28, 28), cmap="gray_r", vmin=0, vmax=1)
    ax.set_title(f"“digit {Pn[j].argmax()}”: {pct(Pn[j].max(), 1)}", fontsize=9)
    ax.axis("off")
plt.suptitle("The 5 noise images the network is most confident about", fontsize=10, y=1.04)
plt.show()

The network doesn't answer “don't know”. Softmax always distributes the full 100% over the 10 digits, because there
is no “not a digit” option, and on noise images the network is still often quite confident (figure above: the noise
images it is most confident about). On noise images the mean confidence is lower than on real images (70.45% versus
97.15%), which is why Hendrycks & Gimpel (2017) use the largest probability itself as a baseline for detecting
unfamiliar images, but it can't be relied on: softmax's probabilities only mean something on the kind of data the
network has learned.

## 10 · The loss is a “landscape”

The loss is a function of all 109,386 parameters. That space can't be drawn, but we can walk along a straight line
in it: from the weights at initialization $\theta_0$ to the trained weights $\theta_1$,
$\theta(\alpha) = (1-\alpha)\,\theta_0 + \alpha\,\theta_1$, and measure the loss along the way (the view of
Goodfellow, Vinyals & Saxe, 2015). In the figure below, the top panel is the cross-entropy (blue, circles), the
bottom one the accuracy (orange, squares, dashed), both on the 10,000 test images.

In [ ]:
alphas = np.linspace(-0.25, 1.5, 15)
path_loss, path_acc = [], []
for a in alphas:
    theta = [[(1 - a) * W0 + a * W1, (1 - a) * b0 + a * b1] for (W0, b0), (W1, b1) in zip(params0, params)]
    Za = forward(theta, Xte)[2]
    path_loss.append(cross_entropy(Za, yte))
    path_acc.append(float((Za.argmax(1) == yte).mean()))
i0, i1 = np.argmin(np.abs(alphas)), np.argmin(np.abs(alphas - 1))
print("α:      ", "  ".join(f"{num(a, 3):>6}" for a in alphas))
print("loss:   ", "  ".join(f"{num(v, 3):>6}" for v in path_loss))
print(
    f"from θ₀ to θ₁, the loss decreases at every step: {bool(np.all(np.diff(path_loss[i0 : i1 + 1]) < 0))};  "
    f"smallest at α = {num(alphas[np.argmin(path_loss)], 3)}"
)
assert np.isclose(path_loss[i0], loss0) and np.isclose(path_loss[i1], loss)
fig, (a, b) = plt.subplots(2, 1, figsize=(8, 4.6), sharex=True)
a.plot(alphas, path_loss, color=BLUE, marker="o", lw=2)
b.plot(alphas, 100 * np.array(path_acc), color=ORANGE, marker="s", ls="--", lw=2)
for ax in (a, b):
    for x0 in (0, 1):
        ax.axvline(x0, color=GREY, lw=1)
a.text(0.02, 2.2, "θ₀: at initialization", fontsize=9)
a.text(1.02, 2.2, "θ₁: trained", fontsize=9)
a.set(ylabel="cross-entropy", title="Walking straight from the initial weights to the trained weights")
b.set(ylabel="correct (%)", xlabel="α  (θ = (1 − α)·θ₀ + α·θ₁)")
fmt_ticks(a.yaxis)
fmt_ticks(b.xaxis, 2)
for ax in (a, b):
    ax.grid(alpha=0.3)
plt.show()

Along this straight line, the loss decreases steadily from $\theta_0$ to $\theta_1$, and the accuracy shoots up in
the middle stretch. Go a little past $\theta_1$ ($\alpha > 1$) and the loss rises again: $\theta_1$ lies close to the
bottom of this slice, though not necessarily at the true bottom of the whole space. Gradient descent (Lesson 05)
doesn't know $\theta_1$ in advance. It only stands where it is, looks at the slope
$\partial\mathcal{L}/\partial\theta$ and steps downhill. The loss function is the very landscape it walks on.

## 11 · Try changing things

The cells below rerun the same functions with other settings. You can change the values and run them again.

### The temperature of softmax
Divide every logit by a number $T > 0$ before the softmax: $T < 1$ makes the probabilities “sharper” (more
confident), $T > 1$ makes them “flatter”. The order of the digits and the accuracy don't change; only the confidence
does. This is also the “temperature” knob when GPT-2 generates text (Lesson 17). The cell below measures the trained
network's test cross-entropy at several temperatures, and the probability of the digit the network picks for the
unsure image #340.

In [ ]:
for T in [0.5, 0.75, 1, 1.5, 2, 4]:
    ZT = Z / T
    assert ((ZT.argmax(1) == yte).mean()) == acc  # the accuracy doesn't change
    print(
        f"T = {num(T, 2):>4}: test loss {num(cross_entropy(ZT, yte), 4)};  image #340: ŷ of the 5 = {pct(softmax(ZT[340])[5], 1)}"
    )

Among these values, the loss is smallest at exactly $T = 1$: the network is already “calibrated”, as the figure in
section 9 shows. For large overconfident networks, Guo et al. (2017) choose $T > 1$ on a validation set
(*temperature scaling*), and that single parameter is enough to fix most of the overconfidence.

### Can MSE really not learn?
Section 8 showed that MSE loses when the network starts out confidently wrong. What about a normal initialization?
MSE's slopes are smaller, so it is only fair to give it a larger learning rate. The cell below trains for one epoch,
3 seeds, and measures on a **validation set** (the last 5,000 images of the training set, `train[55k:]`, as in the
course's later lessons), not on the test set, because we are choosing $\eta$.

In [ ]:
Xa, ya, Xv, yv = Xtr[:55000], ytr[:55000], Xtr[55000:], ytr[55000:]
print(f"training on {num(len(ya), 0)} images, validating on {num(len(yv), 0)} images")
val = {}
for name, g, eta in [("cross-entropy", grad_ce, 0.1), ("MSE", grad_mse, 0.1), ("MSE", grad_mse, 1.0)]:
    r = [
        train(seed=s, eta=eta, epochs=1, grad=g, X=Xa, y=ya, X_eval=Xv, y_eval=yv)[1]["epoch_acc"][0] for s in range(3)
    ]
    val[(name, eta)] = r
    print(
        f"{name:>13}, η = {num(eta, 1)}: validation accuracy {' · '.join(pct(a) for a in r)}  (from {pct(min(r))} to {pct(max(r))})"
    )

At the same $\eta = 0.1$, MSE is below cross-entropy for all three seeds (the same seed means the same
initialization and the same image order), but the two ranges overlap. With $\eta = 1$, MSE's range (91.00–96.30%)
contains that of cross-entropy: the difference is within the variation between seeds. (We haven't tuned $\eta$ for
cross-entropy here; try adding it to the list.) So “MSE can't learn classification” is an overstatement. Hui & Belkin
(2021) trained many large networks with MSE and found that it often matches cross-entropy. The reason cross-entropy
is the default is that it never “switches off” the slope when the network is confidently wrong, and it is a
log-likelihood, so $\hat y$ means a probability.

Try also: change `last_scale` in section 8 to 10 or 100; change `eta`; or replace `grad_mse` with a loss function of
your own (it only needs to return $\partial\mathcal{L}/\partial z$).

## 12 · Limits, and the bridge to the next lesson

- **The loss only says how wrong the network is.** To fix it, the network needs to know which way each weight
  should move: the gradient, computed by backpropagation (Lesson 06), then descending step by step (Lesson 05). The
  `train()` function above already used both as black boxes.
- **Cross-entropy is never 0.** Image #0 is already correct and certain, yet it still has a positive loss, so
  further training keeps pushing the logits up forever. Once the network gets the whole training set right, that
  leads to overconfidence and overfitting (Lesson 08).
- **The same function for every classification problem.** GPT-2 (Lesson 17) was trained with exactly this
  cross-entropy, each position being a classification problem over the next token with 50,257 classes
  ([Radford et al., 2019](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)). Its “uniform guess” baseline is:

In [ ]:
V = 50257  # the number of tokens in GPT-2's vocabulary
print(f"ln {num(V, 0)} = {num(np.log(V), 2)} nats per token  (MNIST: ln 10 = {num(np.log(10), 2)})")
Zv = np.random.default_rng(0).normal(size=(4, V))  # random logits for 4 "positions", only to try log-softmax
assert np.allclose(np.exp(log_softmax(Zv)).sum(1), 1)

Our log-softmax runs unchanged on 50,257 classes: nothing in it depends on the number 10.

## Summary
- Logits are 10 raw scores; dividing by the sum doesn't give probabilities (some are negative, and the order can be
  reversed). Softmax exponentiates, then divides by the sum: 10 positive numbers, summing to 1, in the same order.
- Softmax depends only on the differences between the logits, so subtracting the max before exponentiating costs
  nothing and never overflows; in float16, the naive softmax gives `nan` on more than half of the test set.
- Cross-entropy $= -\ln \hat y_c$, computed directly from the logits with log-softmax. Untrained network: 2.4257,
  close to $\ln 10$; the slope with respect to the logits is $\hat y - y$.
- Training with cross-entropy (SGD, 5 epochs): 97.27% on the test set, loss 0.0865, exactly as in the demo.
- On confidently wrong images, MSE is stuck near its ceiling of 2 and its slope is 88 to 2,903 times smaller; with an
  overconfident initialization, MSE leaves thousands of images stuck, cross-entropy only a few dozen. With a good
  initialization and a larger $\eta$, MSE does learn.
- This network's probabilities match its accuracy on real images, but 52 errors still get ≥ 90%, and noise images
  get high confidence too.

## Review questions
1. Why not use $z_k / \sum_j z_j$ as probabilities?
2. How does softmax change if you add 100 to all 10 logits? And if you multiply all 10 logits by 3?
3. A freshly initialized 10-class classification network has an initial loss of 9. What does that suggest?
4. A network assigns 0.0001 to the correct digit. What is the cross-entropy of that image, and what is the largest
   the MSE can be?
5. Why is the slope of MSE with respect to the logit of the correct digit close to 0 when the network is confidently
   wrong, while that of cross-entropy isn't?
6. A network says 99% for a noise image. Why can't softmax say “don't know”?

<details><summary>Hints and answers</summary>

1. Logits can be negative: the quotients can be negative or larger than 1, and if the sum is negative the order is
   reversed (section 3).
2. Adding: nothing changes, because $e^{100}$ cancels between the numerator and the denominator. Multiplying by 3:
   the differences between the logits stretch, the largest probability goes toward 1, but the order is kept.
3. A uniform guess gives $\ln 10 \approx 2.30$. A loss of 9 means the initial logits are too large (the output layer
   is initialized too “confidently”), or there is a bug in the code; check before training.
4. $-\ln 0.0001 = 4 \ln 10 \approx 9.21$. The MSE is at most 2 for any image.
5. $\partial\mathrm{MSE}/\partial z_c = -2\,\hat y_c\,\mathrm{MSE}$ has the factor $\hat y_c \approx 0$; whereas
   $\partial\mathcal{L}_\text{CE}/\partial z_c = \hat y_c - 1 \approx -1$, because the logarithm undoes the
   exponential.
6. Softmax always distributes the full 100% over the 10 given classes; there is no “not a digit” class, and the
   network has only learned on digits.
</details>

## Further reading
- MLSysBook, [*Neural Computation*](https://mlsysbook.ai/vol1/nn_computation/nn_computation.html), section *Loss functions*:
  cross-entropy for MNIST, the log-sum-exp trick, an initial loss close to $\log 10$.
- Goodfellow, Bengio & Courville, [*Deep Learning*, section 6.2](https://www.deeplearningbook.org/contents/mlp.html):
  softmax outputs, why log-likelihood suits them, the name “softargmax”.
- [Dive into Deep Learning, *Softmax Regression*](https://d2l.ai/chapter_linear-classification/softmax-regression.html):
  cross-entropy from information theory (nats, surprisal).
- M. Nielsen, [*Neural Networks and Deep Learning*, Chapter 3](http://neuralnetworksanddeeplearning.com/chap3.html):
  squared error slows learning down when the network is badly wrong.
- J. S. Bridle (1990), [Probabilistic interpretation of feedforward classification network outputs](https://doi.org/10.1007/978-3-642-76153-9_28).
- CS231n, [Linear classification](https://cs231n.github.io/linear-classify/): the numerical stability of softmax;
  A. Karpathy, [A Recipe for Training Neural Networks](https://karpathy.github.io/2019/04/25/recipe/): checking the loss at initialization.
- C. Guo et al. (2017), [On Calibration of Modern Neural Networks](https://arxiv.org/abs/1706.04599).
- D. Hendrycks & K. Gimpel (2017), [A Baseline for Detecting Misclassified and Out-of-Distribution Examples](https://arxiv.org/abs/1610.02136).
- L. Hui & M. Belkin (2021), [Square Loss vs Cross-Entropy in Classification Tasks](https://arxiv.org/abs/2006.07322).
- I. Goodfellow, O. Vinyals & A. Saxe (2015), [Qualitatively characterizing neural network optimization problems](https://arxiv.org/abs/1412.6544).

**Next lesson:** [Lesson 05 · Gradient descent](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/gradient-descent/):
walking down the loss landscape step by step, following the slope.

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · numpy {np.__version__} · matplotlib {matplotlib.__version__}")